# Modélisation Textuelle - CamemBERT

## Objectif
Classification des produits Rakuten en utilisant CamemBERT, un modèle de transformer pré-entraîné spécialement conçu pour le français, basé sur l'architecture BERT.

## Approche
1. **CamemBERT** : Utilisation du modèle pré-entraîné `camembert-base` de Hugging Face
2. **Fine-tuning** : Adaptation du modèle pour la classification de produits
3. **Tokenisation** : Utilisation du tokenizer spécifique à CamemBERT
4. **Classification** : Ajout d'une couche de classification sur les embeddings du modèle


In [ ]:
# ========== CONFIGURATION GOOGLE COLAB ==========

import os

# Détection de l'environnement Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
   
    from google.colab import drive
    drive.mount('/content/drive')

    
    DATA_DIR = '/content'  
    OUTPUT_DIR = '/content'  
    # Créer les dossiers de sortie
    os.makedirs(os.path.join(OUTPUT_DIR, 'models'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'output'), exist_ok=True)

    print("✅ Mode Colab détecté - Drive monté")
    print(f"   Données : {DATA_DIR}")
    print(f"   Sortie  : {OUTPUT_DIR}")
    print("   Conseil : Runtime > Modifier le type d'exécution > GPU pour accélérer l'entraînement")
else:
    DATA_DIR = '.'
    OUTPUT_DIR = '.'
    print("✅ Mode local - DATA_DIR et OUTPUT_DIR = '.'")

Mounted at /content/drive
✅ Mode Colab détecté - Drive monté
   Données : /content
   Sortie  : /content
   Conseil : Runtime > Modifier le type d'exécution > GPU pour accélérer l'entraînement


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

# Transformers pour CamemBERT
from transformers import (
    CamembertTokenizer, CamembertForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

# Configuration
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Vérifier si GPU disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Bibliothèques importées")
print(f"Device utilisé : {device}")


✅ Bibliothèques importées
Device utilisé : cuda


In [3]:
# Chargement et nettoyage (DATA_DIR et OUTPUT_DIR définis dans la cellule Colab)
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train_update.csv'))
Y_train = pd.read_csv(os.path.join(DATA_DIR, 'Y_train_CVw08PX.csv'))
X_test = pd.read_csv(os.path.join(DATA_DIR, 'X_test_update.csv'))

train_data = pd.merge(X_train, Y_train, left_index=True, right_index=True)

def nettoyer_texte(texte):
    if pd.isna(texte) or texte == 'nan':
        return ''
    texte = str(texte)
    texte = re.sub(r'<[^>]+>', '', texte)
    texte = texte.replace('&nbsp;', ' ')
    texte = texte.replace('&amp;', '&')
    texte = texte.replace('&lt;', '<')
    texte = texte.replace('&gt;', '>')
    texte = texte.replace('&quot;', '"')
    texte = texte.replace('&#39;', "'")
    texte = texte.replace('&eacute;', 'é')
    texte = texte.replace('&egrave;', 'è')
    texte = texte.replace('&ecirc;', 'ê')
    texte = texte.replace('&agrave;', 'à')
    texte = re.sub(r'\s+', ' ', texte)
    return texte.strip()

train_data['texte_complet'] = (
    train_data['designation'].apply(nettoyer_texte) + ' ' +
    train_data['description'].apply(nettoyer_texte)
).str.strip()

X_test['texte_complet'] = (
    X_test['designation'].apply(nettoyer_texte) + ' ' +
    X_test['description'].apply(nettoyer_texte)
).str.strip()

# Préparer les labels
label_to_id = {label: idx for idx, label in enumerate(sorted(train_data['prdtypecode'].unique()))}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_labels = len(label_to_id)

print(f"✅ Données chargées")
print(f"Nombre de classes : {num_labels}")
print(f"Exemples de labels : {list(label_to_id.items())[:5]}")


✅ Données chargées
Nombre de classes : 27
Exemples de labels : [(np.int64(10), 0), (np.int64(40), 1), (np.int64(50), 2), (np.int64(60), 3), (np.int64(1140), 4)]


In [4]:
# Charger le tokenizer et le modèle CamemBERT
model_name = "camembert-base"

print(f"Chargement du tokenizer {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Chargement du modèle {model_name}...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="single_label_classification"
)
model.to(device)

print(f"✅ Modèle et tokenizer chargés")
print(f"Taille du vocabulaire : {len(tokenizer)}")


Chargement du tokenizer camembert-base...


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Chargement du modèle camembert-base...


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Modèle et tokenizer chargés
Taille du vocabulaire : 32005


In [6]:
# Classe Dataset personnalisée
class ProductDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Préparer les données
X_text = train_data['texte_complet']
y = train_data['prdtypecode'].map(label_to_id)

# Division train/validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {len(X_train_split)}, Validation : {len(X_val_split)}")

# Créer les datasets
train_dataset = ProductDataset(X_train_split, y_train_split, tokenizer)
val_dataset = ProductDataset(X_val_split, y_val_split, tokenizer)

print("✅ Datasets créés")


Train : 67932, Validation : 16984
✅ Datasets créés


In [7]:
# Fonction de calcul des métriques
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_weighted = f1_score(labels, predictions, average='weighted')
    f1_macro = f1_score(labels, predictions, average='macro')

    return {
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
        'f1_macro': f1_macro
    }

# Arguments d'entraînement
training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, 'models', 'camembert_output'),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=100,
    eval_strategy="epoch",  # Note: dans transformers 4.x, c'est "eval_strategy" et non "evaluation_strategy"
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # Utiliser la précision mixte si GPU disponible
)

# Créer le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("✅ Trainer configuré")


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer configuré


In [10]:
# Entraînement
print("Début de l'entraînement...")

trainer.train()

print("✅ Entraînement terminé")


Début de l'entraînement...


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.533973,0.480562,0.858632,0.858077,0.833004
2,0.332446,0.417686,0.885068,0.884773,0.869131
3,0.236459,0.404359,0.893959,0.893468,0.879559


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

✅ Entraînement terminé


In [11]:
# Évaluation sur la validation
print("Évaluation sur le set de validation...")
eval_results = trainer.evaluate()
print("\n" + "="*80)
print("RÉSULTATS SUR LA VALIDATION")
print("="*80)
for key, value in eval_results.items():
    if 'f1' in key or 'accuracy' in key or 'loss' in key:
        print(f"{key}: {value:.4f}")


Évaluation sur le set de validation...



RÉSULTATS SUR LA VALIDATION
eval_loss: 0.4044
eval_accuracy: 0.8940
eval_f1_weighted: 0.8935
eval_f1_macro: 0.8796


In [12]:
# Prédictions sur le test
print("Génération des prédictions sur X_test...")

# Fonction pour prédire
def predict_texts(texts, model, tokenizer, device, batch_size=16):
    model.eval()
    predictions = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts.iloc[i:i+batch_size].tolist()

        encodings = tokenizer(
            batch_texts,
            truncation=True,
            padding='max_length',
            max_length=512,
            return_tensors='pt'
        )

        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits
            batch_predictions = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(batch_predictions)

    return predictions

# Prédictions
y_test_pred_ids = predict_texts(X_test['texte_complet'], model, tokenizer, device)
y_test_pred = [id_to_label[pred_id] for pred_id in y_test_pred_ids]

print(f"✅ {len(y_test_pred)} prédictions générées")
print(f"Distribution des prédictions :")
print(pd.Series(y_test_pred).value_counts().head(10))


Génération des prédictions sur X_test...
✅ 13812 prédictions générées
Distribution des prédictions :
2583    1670
2522     857
2280     835
1300     822
2060     794
1280     790
1560     783
2403     753
1160     661
1920     655
Name: count, dtype: int64


In [13]:
# Sauvegarder le modèle et les résultats (compatible Colab / local)
models_dir = os.path.join(OUTPUT_DIR, 'models')
output_dir = os.path.join(OUTPUT_DIR, 'output')
os.makedirs(models_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Sauvegarder le modèle fine-tuné
model_path = os.path.join(models_dir, 'camembert_finetuned')
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Sauvegarder les mappings de labels
joblib.dump(label_to_id, os.path.join(models_dir, 'camembert_label_to_id.pkl'))
joblib.dump(id_to_label, os.path.join(models_dir, 'camembert_id_to_label.pkl'))

# Sauvegarder les prédictions
predictions_df = pd.DataFrame({
    'productid': X_test['productid'],
    'prdtypecode': y_test_pred
})
predictions_df.to_csv(os.path.join(output_dir, 'predictions_camembert.csv'), index=False)

print(f"✅ Modèle sauvegardé : {model_path}")
print(f"✅ Prédictions sauvegardées : {os.path.join(output_dir, 'predictions_camembert.csv')}")
print("\n" + "="*80)
print("RÉSUMÉ")
print("="*80)
print(f"Accuracy (validation) : {eval_results.get('eval_accuracy', 'N/A'):.4f}")
print(f"F1-score weighted (validation) : {eval_results.get('eval_f1_weighted', 'N/A'):.4f}")
print(f"F1-score macro (validation) : {eval_results.get('eval_f1_macro', 'N/A'):.4f}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle sauvegardé : /content/models/camembert_finetuned
✅ Prédictions sauvegardées : /content/output/predictions_camembert.csv

RÉSUMÉ
Accuracy (validation) : 0.8940
F1-score weighted (validation) : 0.8935
F1-score macro (validation) : 0.8796
